# Meet the Bash Agent

In the previous modules, you built agents from scratch, gave them tools, and learned how to evaluate their performance. Now it's time to take things a step further: **customizing an agent to make it truly your own.**

But before we can customize anything, we need to understand what we're working with. In this notebook, you'll get hands-on experience with a **Bash Agent**—an AI assistant that can understand natural language requests and translate them into shell commands.

## What is a Bash Agent?

Think about how you interact with your computer's terminal. You type precise commands like `ls -la` or `grep -r "error" ./logs`. These commands are powerful, but they require you to remember exact syntax, flags, and options.

A Bash Agent flips this paradigm. Instead of memorizing arcane command syntax, you simply tell the agent what you want:
- *"Show me all the Python files in this directory"*
- *"Find the largest files here"*
- *"Create a new folder called 'experiments' and list what's inside"*

The agent understands your intent and translates it into the appropriate shell commands. It's like having a knowledgeable colleague who's fluent in bash sitting right next to you.

## Why This Agent?

We chose a Bash Agent for this customization module for several important reasons:

1. **Observable Behavior**: Shell commands produce clear, measurable outputs. When the agent runs `ls`, you can see exactly what it did and whether it succeeded.

2. **Safety Boundaries**: The agent operates within a restricted set of allowed commands—no `rm -rf /` disasters here! This teaches an important lesson about designing safe, constrained agents.

3. **Real-World Applicability**: Many AI engineers use agents like this daily to automate repetitive system tasks, explore codebases, and manage files.

4. **Clear Improvement Opportunities**: As you'll see, the agent sometimes makes suboptimal choices. These "rough edges" are perfect targets for the customization techniques you'll learn later in this module.

---

## Learning Objectives

By the end of this notebook, you will be able to:

- ✅ Understand the architecture of a ReAct-style agent built with LangGraph
- ✅ Explain the role of human-in-the-loop confirmation for tool execution
- ✅ Interact with the Bash Agent to observe its behavior and decision-making
- ✅ Identify opportunities where the agent's performance could be improved

Let's dive in!

## 1. Environment Setup

First, let's set up our environment. This cell installs the required dependencies and loads our API keys from environment files.

> 💡 **Note**: If you haven't configured your secrets yet, head back to the [Setting up Secrets](../secrets.md) page before continuing.


In [ ]:
# Dependencies are provided by the locked project environment.
from dotenv import load_dotenv
_ = load_dotenv("../../variables.env")
_ = load_dotenv("../../secrets.env")

## 2. Understanding the Architecture

Before we build the agent, let's understand the key components we're importing. This agent is built using **LangGraph**, a framework for creating stateful, multi-step AI applications.

| Component | Purpose |
|-----------|---------|
| `create_react_agent` | Creates a ReAct (Reason + Act) agent that can think, use tools, and respond |
| `InMemorySaver` | Stores conversation history so the agent remembers previous interactions |
| `ChatOpenAI` | A chat model client compatible with OpenAI-style APIs (including NVIDIA NIM) |
| `Config` | Our custom configuration class that defines model settings and the system prompt |
| `Bash` | Our custom tool that executes bash commands with safety guardrails |
| `get_skill` | Tool to load skills that provide structured workflows for complex tasks |
| `list_available_skills` | Tool to list all available skills the agent can use |

The **ReAct pattern** is a powerful agent architecture where the model:
1. **Reasons** about what it needs to do
2. **Acts** by calling a tool
3. **Observes** the result
4. Repeats until the task is complete


In [ ]:
from typing import Dict
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI

from bash_agent.config import Config
from bash_agent.bash import Bash
from bash_agent.skills import get_skill, list_available_skills
import os

os.environ["LANGCHAIN_TRACING_V2"] = "false"

## 3. Human-in-the-Loop: The Safety Net

Here's where things get interesting. Running arbitrary shell commands is inherently risky. What if the agent misunderstands you and tries to delete important files?

We solve this with a **Human-in-the-Loop (HITL)** pattern. The `ExecOnConfirm` class wraps our Bash tool and requires your explicit approval before executing any command.

```
User: "List all files"
   ↓
Agent thinks: "I should run 'ls -la'"
   ↓
ExecOnConfirm: "Execute 'ls -la'? [y/N]:"
   ↓
User types 'y' → Command runs
User types 'n' → Command blocked, agent informed
```

This pattern is essential for any agent that can take actions in the real world. Let's look at the implementation.


### 📋 Exercise

Implement the HITL confirmation gate for command execution in the below cell.

If the user confirms via `self._confirm_execution(cmd)`, return `self.bash.exec_bash_command(cmd)`. Otherwise, return a dictionary with `"error"` set to `"User declined."` so the agent knows the command was rejected.

<details>
<summary>💡 NEED SOME HELP?</summary>

---

```python
if self._confirm_execution(cmd):
    return self.bash.exec_bash_command(cmd)
return {"error": "The user declined the execution of this command."}
```
</details>


In [ ]:
class ExecOnConfirm:
    """
    A wrapper around the Bash tool that asks for user confirmation before executing any command.
    """

    def __init__(self, bash: Bash):
        self.bash = bash

    def _confirm_execution(self, cmd: str) -> bool:
        """Ask the user whether the suggested command should be executed."""
        return input(f"    ▶️   Execute '{cmd}'? [y/N]: ").strip().lower() == "y"

    def exec_bash_command(self, cmd: str) -> Dict[str, str]:
        """Execute a bash command after confirming with the user."""
        if self._confirm_execution(cmd):
            return self.bash.exec_bash_command(cmd)
        return {"error": "The user declined the execution of this command."}

Notice how the wrapper:
1. Prompts you with the exact command the agent wants to run
2. Only proceeds if you type `y` (lowercase)
3. Returns an error message to the agent if you decline—so it knows not to keep trying

This is a simple but effective safety mechanism. In production systems, you might replace this with approval workflows, audit logs, or policy engines.


## 4. Loading the Configuration

The `Config` class (defined in `bash_agent/config.py`) centralizes all the settings for our agent. Let's load it and see what it contains.

Key configuration elements include:
- **Model settings**: Which LLM to use, the API endpoint, and sampling parameters
- **Allowed commands**: A whitelist of safe bash commands the agent can execute
- **System prompt**: Instructions that define the agent's personality and constraints

Run the cell below to load the configuration:


In [ ]:
config = Config()

## 5. Creating the LLM Client

Now we create the language model client. We're using `ChatOpenAI` from LangChain, which works with any OpenAI-compatible API—including NVIDIA's NIM endpoints.

The configuration pulls from our `Config` class:
- **Model**: `nvidia/nvidia-nemotron-nano-9b-v2` — a compact, efficient model with strong reasoning capabilities
- **Temperature**: `0.1` — low temperature for more deterministic, predictable outputs
- **Top-p**: `0.95` — nucleus sampling to maintain some response diversity


In [ ]:
# Create the client
llm = ChatOpenAI(
    model=config.llm_model_name,
    openai_api_base=config.llm_base_url,
    openai_api_key=config.llm_api_key,
    temperature=config.llm_temperature,
    top_p=config.llm_top_p,
)

## 6. Creating the Bash Tool

The `Bash` class (from `bash_agent/bash.py`) is the heart of our agent's capabilities. It's responsible for:

1. **Command validation**: Checking that requested commands are in the allowlist
2. **Security checks**: Blocking command injection patterns (like backticks or `$` variables)
3. **Execution**: Running approved commands via subprocess
4. **State tracking**: Keeping track of the current working directory across commands

Take a look at `bash_agent/bash.py` to see the full implementation. For now, let's instantiate the tool:


In [ ]:
# Create the tool
bash = Bash(config)

## 7. Assembling the Agent

Now we bring everything together! The `create_react_agent` function from LangGraph takes:

- **model**: Our LLM client
- **tools**: A list of callable functions (wrapped in `ExecOnConfirm` for safety), plus skill tools
- **prompt**: The system prompt that defines the agent's behavior
- **checkpointer**: Memory storage for conversation continuity

Notice how we pass `ExecOnConfirm(bash).exec_bash_command` as the tool—this ensures every command goes through our human approval step.

We also include **skill tools** from the [Superpowers framework](https://github.com/obra/superpowers):
- `get_skill`: Loads a skill that provides structured workflows for complex tasks
- `list_available_skills`: Lists all available skills the agent can use

Skills include workflows for systematic debugging, test-driven development, brainstorming, and planning.

### 📋 Exercise

Assemble the ReAct agent from model, tools, and prompt.

LangGraph’s `create_react_agent` wires together the Reason-Act-Observe loop from Module 1. Implement `agent` with `model` set to `llm`, `tools` set to a list containing your `ExecOnConfirm`-wrapped bash command, and `prompt` set to `config.system_prompt`.

<details>
<summary>💡 NEED SOME HELP?</summary>

---

```python
agent = create_react_agent(
    model=llm,
    tools=[
        ExecOnConfirm(bash).exec_bash_command,
        get_skill,              # Load skills for structured workflows
        list_available_skills,  # List available skills
    ],
    prompt=config.system_prompt,
    checkpointer=InMemorySaver(),
)
```
</details>

In [ ]:
# Create the agent with bash tool and skill tools
agent = create_react_agent(
    model=llm,
    tools=[
        ExecOnConfirm(bash).exec_bash_command,
        get_skill,              # Load skills for structured workflows
        list_available_skills,  # List available skills
    ],
    prompt=config.system_prompt,
    checkpointer=InMemorySaver(),
)
print("[INFO] Type 'quit' at any time to exit the agent loop.\n")
print("[INFO] This agent has access to Superpowers skills. Try asking it to 'list available skills'!\n")

## 8. Interacting with the Agent

It's showtime! 🎬 The cell below starts an interactive loop where you can chat with your Bash Agent.

### How to Use It

1. **Run the cell** to start the agent
2. **Type your request** in natural language (e.g., "What files are in this directory?")
3. **Review the command** when prompted — type `y` to approve or `n` to decline
4. **Observe the result** and continue the conversation
5. **Type `quit`** when you're done

### Things to Try

Here are some requests to test the agent's capabilities:

| Request | What It Tests |
|---------|---------------|
| "List all files here" | Basic command translation |
| "Show me the contents of config.py" | File reading |
| "Find all Python files in this directory" | Pattern matching with `find` |
| "How big are the files in this folder?" | Using `du` for disk usage |
| "Create a folder called 'test' and show me what's inside" | Multi-step task |
| "List available skills" | Skill discovery |
| "Load the systematic-debugging skill" | Loading a skill workflow |
| "I need to debug an issue, what skill should I use?" | Skill-guided task |

### What to Watch For

As you interact with the agent, pay attention to:
- **Command choices**: Does it pick the most efficient command?
- **Error handling**: How does it respond when something goes wrong?
- **Clarity**: Are its explanations helpful?

These observations will inform the customization work in the next notebook!


---

> ⚠️ **Interactive Cell**: The cell below will wait for your input. Make sure to type `quit` when you're done, or use the stop button to interrupt the kernel.


### 📋 Exercise

Send the user’s message into the agent loop.

`agent.invoke()` kicks off the full ReAct cycle — reason, propose a tool call, wait for HITL approval, observe the result, and repeat. Implement `result` by invoking the agent with a single message where `"role"` is `"user"` and `"content"` is the user input variable.

<details>
<summary>💡 NEED SOME HELP?</summary>

---

```python
result = agent.invoke(
    {"messages": [{"role": "user", "content": user}]},
    config={"configurable": {"thread_id": "cli"}},  # one ongoing conversation
)
```
</details>


In [ ]:
# The main loop
while True:
    user = input(f"['{bash.cwd}' 🙂] ").strip()

    if user.lower() == "quit":
        print("\n[🤖] Shutting down. Bye!\n")
        break
    if not user:
        continue

    # Always tell the agent where the current working directory is to avoid confusions.
    user += f"\n Current working directory: `{bash.cwd}`"
    print("\n[🤖] Thinking...")

    # Run the agent's logic and get the response.
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user}]},
        config={"configurable": {"thread_id": "cli"}},  # one ongoing conversation
    )
    # Show the response (without the thinking part, if any)
    response = result["messages"][-1].content.strip()

    if "</think>" in response:
        response = response.split("</think>")[-1].strip()

    if response:
        print(response)
        print("-" * 80 + "\n")

## 📋 Exercise: Discover Agent Limitations

While running your agent, ask it a question on a custom CLI command as an edge case. How does the agent respond? You may notice the agent is great at executing native bash commands, but custom CLI commands may be outside of its domain. 

For example, an example query could be: `Create a new project using langgraph cli commands`. You may see: 

```
I can help you set up a directory structure for your langgraph project using allowed commands, but I can't directly execute langgraph CLI commands since they aren't in my allowed tool list. Would you like me to:

1. Create a new directory for your project using `mkdir`?
2. Navigate into it with `cd`?
3. Or do you need specific langgraph CLI commands that I can't execute directly?

Let me know how you'd like to proceed!
```

Sounds like we need to add this domain expertise to the agent somehow! 

## 🎯 Reflection: What Did You Notice?

Before moving on, take a moment to reflect on your experience with the agent:

1. **Did the agent always choose the best command?** Sometimes a simpler command would suffice, or a more powerful one would be more appropriate.

2. **How did it handle ambiguity?** When your request wasn't perfectly clear, did it ask for clarification or make assumptions?

3. **Was the output format helpful?** Did it present information in a way that was easy to understand?

4. **Did it ever fail?** What happened when a command didn't work as expected?

These observations are valuable! They highlight areas where the agent's behavior could be improved—exactly what we'll tackle in the customization exercises.

---

## Alternative: Running via Command Line

Prefer working in a terminal? You can run this same agent as a standalone CLI application.

**From the project root, run:**

```bash
cd /project/code/4-agent-customization && python -m bash_agent.main_langgraph
```

This uses the same code but in a more traditional terminal experience. There's also a "from scratch" implementation that doesn't use LangGraph:

```bash
cd /project/code/4-agent-customization && python -m bash_agent.main_from_scratch
```

Comparing these two implementations is a great way to understand what LangGraph provides out of the box!

---

## What's Next?

Now that you've gotten hands-on experience with the Bash Agent and identified some areas for improvement, you're ready to learn how to **customize** it.

In the next notebook, we'll explore techniques for improving agent behavior through:
- **Synthetic Data Generation (SDG)**: Creating training data from agent interactions
- **Reinforcement Learning (RL)**: Using feedback to optimize agent decisions

Head over to the next lesson to continue your journey! 🚀